In [12]:
import warnings
import logging

warnings.filterwarnings("ignore")
logging.getLogger("LiteLLM").setLevel(logging.ERROR)


In [13]:
import litellm

litellm.suppress_debug_info = True

In [14]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [ ]:

# Now import LiteLLM normally
from litellm import completion

response_openai = completion(
    model="groq/openai/gpt-oss-120b",
    messages=[{"role": "user", "content": "Explain RAG in one sentence."}]
)

print("OpenAI:  ", response_openai.choices[0].message.content)

In [ ]:
import os
from litellm import completion

response = completion(
    model="groq/openai/gpt-oss-120b",
    messages=[
        {
            "role": "user",
            "content": "Explain LangGraph architecture"
        }
    ],
    api_key=os.getenv("GROQ_API_KEY")
)

print(response.choices[0].message.content)

In [ ]:
prompt = "Explain RAG in one sentence?"

providers = [
    ("🔵 OpenAI", "gpt-4o-mini"),
    ("🟢 Groq", "groq/openai/gpt-oss-120b"),
    ("🟡 Gemini", "gemini/gemini-3.8-flash")
]

for label, model in providers:
    try:
        r = completion(model=model, messages=[{"role": "user", "content": prompt}])
        print(f"{label:<15} {r.choices[0].message.content[:80]}")
        print("="*80)
    except Exception as e:
        print(f"{label:<15} ❌ {type(e).__name__}")

### Part 4: Automatic Fallbacks

In [ ]:
from litellm import completion

response = completion(
    model="gemini/gemini-3.8-flash",
    messages=[
        {
            "role": "user",
            "content": "What is LLM Gateway?"
        }
    ],
    fallbacks=[
        "groq/openai/gpt-oss-120b",
        "gpt-4o-mini"
    ]
)

print(f"🤖 {response.model} - {response.choices[0].message.content}")

### 💰 Part 5: Cost Tracking - Know Where Your Money Goes

In [ ]:
from litellm import completion, completion_cost, fallbacks

response = completion(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Write a haiku about AI."}],
    fallbacks=[
        "gemini/gemini-3.8-flash",
        "groq/openai/gpt-oss-120b"
    ]
)

cost = completion_cost(completion_response=response)

print("Model: ", response.model)
print("Response: ", response.choices[0].message.content)
print("\nInput tokens: ", response.usage.prompt_tokens)
print("Output tokens: ", response.usage.completion_tokens)
print(f"Cost:  ${cost:.8f}")

### Part 6: Caching - Don't pay twice for the same question

In [ ]:
import litellm

litellm.callbacks = []
litellm.success_callback = []
litellm.failure_callback = []
litellm._async_success_callback = []
litellm._async_failure_callback = []

litellm.cache = None

print(f"✅ LiteLLM state reset - ready for clean caching demo")

In [ ]:
import litellm
import time
from litellm import completion
from litellm.caching import Cache

litellm.cache = Cache(type="local")
prompt = "What does LLM stands for? Answer in one line."
start = time.time()
r1 = completion(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": prompt}
    ],
    caching=True
)
t1 = time.time() - start
print(f"❄️ First Call (API): {t1:.2f}s - {r1.choices[0].message.content}")

#-----------------------------------------------------------------------------

start = time.time()
r2 = completion(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": prompt}
    ],
    caching=True
)
t2 = time.time() - start
print(f"❄️ Second Call (cache): {t2:.2f}s - {r2.choices[0].message.content}")
print(f"\n 🚀 Speedup: {t1/t2:.1f}x faster, and ZERO cost one the second call!")

### 🚏 Part 7: Smart Routing - Right Model for the Right job
🔵 GPT-4o-mini

GPT-4o-mini
(Best for: High-volume, simple tasks.)

Use it for:

* Text classification.
* Query understanding.
* Intent detection.
* Text summarization.
* Simple SQL generation.
* Extracting structured information.
* Basic chatbot conversations.
* Input validation.

🟢 Groq – GPT-OSS-120B

GPT-OSS-120B
(Best for: Reasoning, planning and agentic workflows.)

Use it for:

* Complex task decomposition.
* Multi-step reasoning.
* Planning agent execution.
* Coding and debugging.
* Tool selection.
* Research agents.
* Mathematical reasoning.
* Structured agent workflows.

🟡 Gemini – Gemini 3.8 Flash

Gemini 3.8 Flash
(Best for: Complex workflows, coding and multimodal understanding.)

Use it for:

* Advanced code generation.
* Repository analysis.
* Complex debugging.
* Long-document analysis.
* Image understanding.
* Video understanding.
* PDF processing.
* Multi-step agent orchestration.
* Complex planning.

In [ ]:
import os
from litellm import Router

model_list = [
    {
        "model_name": "fast-cheap",
        "litellm_params": {
            "model": "gpt-4o-mini",
            "api_key": os.getenv("OPENAI_API_KEY")
        }
    },
    {
        "model_name": "smart-coding",
        "litellm_params": {
            "model": "groq/openai/gpt-oss-120b",
            "api_key": os.getenv("GROQ_API_KEY")
        }
    },
    {
        "model_name": "balanced",
        "litellm_params": {
            "model": "gemini/gemini-3.8-flash",
            "api_key": os.getenv("GEMINI_API_KEY")
        }
    }
]

router = Router(model_list=model_list)

fast_response =  router.completion(
   model="fast-cheap",
   messages=[{"role": "user", "content": "Summarize: AI is changing software."}]
)

code_response =  router.completion(
   model="smart-coding",
   messages=[{"role": "user", "content": "Write a python function to reverse a string?"}]
)

print(f"⚡️ Fast/Cheap (OpenAI): {fast_response.choices[0].message.content[:150]}")
print(f"🧠 Smart Coding (Groq): {code_response.choices[0].message.content[:1000]}")


### 🔁 Part 8: Load Balancing Across Multiple API Keys

In [ ]:
import os
from litellm import Router

model_list = [
    {
        "model_name": "gpt-pool",
        "litellm_params": {
            "model": "gpt-4o-mini",
            "api_key": os.getenv("OPENAI_API_KEY")
        },
        "model_info": {"id": "openai-gpt4omini"}
    },
    {
        "model_name": "gpt-pool",
        "litellm_params": {
            "model": "groq/openai/gpt-oss-120b",
            "api_key": os.getenv("GROQ_API_KEY")
        },
        "model_info": {"id": "groq-openai-gptoss120b"}
    }
]

router = Router(
    model_list=model_list,
    routing_strategy="simple-shuffle"
)

print(f"{'Request':<10}{'Deployment Picked':<22}{'Latency':<12}{'Response':<40}")
print("-"*84)

for i in range(6):
    r = router.completion(
        model="gpt-pool",
        messages=[{
            "role": "user",
            "content": f"Say hello, request {i+1}"
        }],
    )
    deployment_id = r._hidden_params.get("model_id", "unknown")
    latency = r._response_ms
    answer = r.choices[0].message.content[:35]
    print(f"#{i+1:<9} {deployment_id:<22} {latency:<6.0f}ms {answer}")


In [ ]:
from typing import Counter

router = Router(
    model_list=model_list,
    routing_strategy="least-busy"
)

hits = Counter()

for i in range(8):
    r = router.completion(
        model="gpt-pool",
        messages=[{"role": "user", "content": f"Say 'OK' #{i}"}],
        max_tokens=5
    )
    hits[r._hidden_params.get("model_id", "?")] += 1
    print(f"Request {i+1} -> {r._hidden_params.get('model_id', '?')}")

print(f"❇️ Distribution:")
for k, v in hits.most_common():
    print(f"    {k}: {' ' * v} ({v})")


### 🚀 Strategy 2: (latency-based-routing)

In [ ]:
import os
import litellm
from typing import Counter
from litellm import Router

litellm.set_verbose = False

model_list = [
    {
        "model_name": "gpt-pool",
        "litellm_params": {
            "model": "openai/gpt-4o-mini",
            "api_key": os.getenv("OPENAI_API_KEY")
        },
        "model_info": {"id": "openai-gpt4omini"}
    },
    {
        "model_name": "gpt-pool",
        "litellm_params": {
            "model": "groq/openai/gpt-oss-120b",
            "api_key": os.getenv("GROQ_API_KEY"),
        },
        "model_info": {"id": "groq-openai-gptoss120b"}
    },
    {
        "model_name": "gpt-pool",
        "litellm_params": {
            "model": "gemini/gemini-3.8-flash",
            "api_key": os.getenv("GEMINI_API_KEY"),
        },
        "model_info": {"id": "gemini-gemini38flash"}
    }
]

router = Router(
    model_list=model_list,
    routing_strategy="cost-based-routing", # Execute acompletion(..) use while using this routing strategy
    num_retries=2
)

hit = Counter()

for i in range(8):
    r = await router.acompletion(
        model="gpt-pool",
        messages=[{"role": "user", "content": f"Say 'OK' #{i}"}],
        max_tokens=5
    )
    hit[r._hidden_params.get("model_id", "?")] += 1
    print(f"Request {i+1} -> {r._hidden_params.get('model_id', '?')}")

print(f"❇️ Distribution:")
for k, v in hit.most_common():
    print(f"    {k}: {' ' * v} ({v})")

### 🔗 Part 9: Integrating the Gateway with LangChain
*LangChain* is the orchestration (agents, chains, RAG) + *LiteLLM* is the unified LLM backend.

LangChain has a built-in **ChatLiteLLM** wrapper - drop it in like any other chat model.

In [ ]:
!pip install -q langchain-litellm

In [ ]:
from langchain_litellm import ChatLiteLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Build a chat model that talk though LiteLLM
llm = ChatLiteLLM(model="gpt-4o-mini", temperature=0.3)

# A standard langchain prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI tutor named KrishnaGPT, Be concise."),
    ("user", "{question}")
])

# Compose with LCEL (read about it) - Same syntax as native LangChain
chain = prompt | llm | StrOutputParser()

answer = chain.invoke({"question": "What is an LLM Gateway in 3 bullets?"})
print(answer)

In [ ]:
! uv add -r requirements.txt --upgrade

### 🤖 Part 10: A Real Example - Multi-Provider LangChain Chain with Fallbacks

Let's combine everything a LangChain chain that uses Claude as primary, with GPT and Groq as fallbacks.

In [ ]:
from langchain_litellm import ChatLiteLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Build a chat model that talk though LiteLLM
primary = ChatLiteLLM(model="gpt-4o-mini", temperature=0.3)

# Fallbacks (any LangChain compatible model)
fallback_1 = ChatLiteLLM(model="groq/openai/gpt-oss-120b", temperature=0.3)
fallback_2 = ChatLiteLLM(model="gemini/gemini-3.8-flash", temperature=0.3)

# LangChain's .with_fallbacks() chains them together
robust_llm = primary.with_fallbacks([fallback_1, fallback_2])

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert AI engineer. Always reply in JSON: {{\"answer\": ...}}"),
    ("user", "{question}")
])

chain = prompt | robust_llm | StrOutputParser()

result = chain.invoke({
    "question": "What are the top 3 benefits of an LLM Gateway?"
})

print(result)



### Part - 11: A Mini End-To-End Demo - Smart Router for a Chatbot

Let's build a tiny __task-aware chatboat__ that:

- Decide what kind of question the user is asking (code, summary or general)
- Routes to the right model accordingly.
- Falls back if the chosen model fails.
- Logs the cost and latency.

In [ ]:
import time
import os
from litellm import completion, completion_cost, Router

def classify_task(user_query: str) -> str:
    """Cheap classifier - use the faster model to decide routing"""
    cls = completion(
        model="gpt-4o-mini",
        messages=[{
            "role": "user",
            "content": (
                "Classify the following query into EXACTLY one word:"
                f"'code', 'summary', or 'general'. Query: {user_query}\n\nAnswer:"
            )
        }],
        max_tokens=5
    )
    return cls.choices[0].message.content.strip().lower()

def call_with_fallback(model_chain, messages):
    last_error = None
    for model in model_chain:
        try:
            return completion(
                model=model, 
                messages=messages,
                fallbacks=[
                    "gpt-4o-mini",
                    "groq/openai/gpt-oss-120b"
                ]
            )
        except Exception as e:
            print(f"⚠️ Model {model} failed ({type(e).__name__}), trying next!")
            last_error = e
            continue
    return last_error
    

async def smart_chat(user_query: str):
    """Choose the task classification and call the respective model for query"""
    classify = classify_task(user_query)
    model_list = [
        {
            "model_name": "summary",
            "litellm_params": {
                "model": "gpt-4o-mini",
                "api_key": os.getenv("OPENAI_API_KEY")
            }
        },
        {
            "model_name": "code",
            "litellm_params": {
                "model": "groq/openai/gpt-oss-120b",
                "api_key": os.getenv("GROQ_API_KEY")
            }
        },
        {
            "model_name": "general",
            "litellm_params": {
                "model": "gemini/gemini-3.8-flash",
                "api_key": os.getenv("GEMINI_API_KEY")
            }
        }
    ]

    router = Router(
        model_list=model_list,
        routing_strategy="least-busy", # Execute acompletion(..) use while using this routing strategy
        num_retries=3,
        allowed_fails=2,
        cooldown_time=60,
        fallbacks=[
            {"general": ["summary"]}
        ]
    )
    start = time.time()
    response = router.completion(
        model=classify,
        messages=[{
            "role": "user",
            "content": user_query
        }],
    )
    latency = time.time() - start

    try:
        cost = completion_cost(completion_response=response)
        cost_str = f"${cost:.02f}"
    except Exception as e:
        print(f"ERROR: {e}")
        cost_str = "n/a"

    answer = "N/A"
    if response is not None:
        answer = response.choices[0].message.content
    else:
        answer = "N/A"
    return {
        "detected_task": classify,
        "model_used": response.model,
        "latency_sec": round(latency, 2),
        "cost_usd": cost_str,
        "answer": answer
    }
    

queries = [
    "Write a python function to complete Fibonacci numbers",
    "Summarize the importance of attention mechanism in 2 sentences",
    "Tell me a fun fact about elephants"
]

for query in queries:
    print("=" * 70)
    print(f"❓q: {query}")
    try:
        result = await smart_chat(query)
        print(f"Task: {result.get('detected_task')}")
        print(f"Model: 🤖{result.get('model_used')}")
        print(f"Answer: {result.get('answer')[:200]}...")
        print(f"Cost: 💵 {result.get('cost_usd')}")
        print(f"Latency: ⏱️{result.get('latency_sec')}s")
    except Exception as e:
        print("ERROR: ", e)

❓q: Write a python function to complete Fibonacci numbers
ERROR: This model isn't mapped yet. model=openai/gpt-oss-120b, custom_llm_provider=openai. Add it here - https://github.com/BerriAI/litellm/blob/main/model_prices_and_context_window.json.
Task: code
Model: 🤖openai/gpt-oss-120b
Answer: Here’s a compact, well‑documented Python implementation that can:

* Return the *n*‑th Fibonacci number (`fib(n)`),  
* Generate a list of the first *n* Fibonacci numbers (`fib_series(n)`),  
* Use th...
Cost: 💵 n/a
Latency: ⏱️3.44s
❓q: Summarize the importance of attention mechanism in 2 sentences
Task: summary
Model: 🤖gpt-4o-mini-2024-07-18
Answer: The attention mechanism is crucial in neural networks as it allows models to focus on relevant parts of the input data, improving performance on tasks involving sequential or structured information, s...
Cost: 💵 $0
Latency: ⏱️2.02s
❓q: Tell me a fun fact about elephants
Task: general
Model: 🤖gpt-4o-mini-2024-07-18
Answer: Elephants are known for their 